# Phase 6: Metrics Agent (Subscriber + Publisher)

This notebook subscribes to spectator events and publishes final KPI metrics at halftime end.

In [ ]:
# Cell 2 purpose: Import modules and ensure src is available from notebooks/.
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path('../src').resolve()))

from simulated_city import mqtt
from simulated_city.config import load_config
from simulated_city.metrics import (
    MetricsAggregatorState,
    enforce_final_scenario_policies,
    finalize_kpi_payload,
    record_spectator_event,
)
from simulated_city.topic_schema import topic_kpi_metrics, topic_spectator_events

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


In [15]:
# Cell 3 purpose: Load config, create metrics aggregator, and connect MQTT client.
app_config = load_config()
if app_config.halftime is None:
    raise ValueError('Missing halftime section in config.yaml')

enforce_final_scenario_policies(
    missed_kickoff_timestamp_s=900,
    external_disruptions_enabled=False,
    group_coordination_share=0.2,
)
state = MetricsAggregatorState(halftime_duration_s=900)

spectator_topic = topic_spectator_events()
kpi_topic = topic_kpi_metrics()
mqtt_client = mqtt.connect_mqtt(app_config.mqtt, client_id_suffix='metrics-agent')

print(f'Connected to MQTT broker at {app_config.mqtt.host}:{app_config.mqtt.port}')
print(f'Subscribed topic: {spectator_topic}')
print(f'Publish topic: {kpi_topic}')

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


Connected to MQTT broker at 127.0.0.1:1883
Subscribed topic: stadium/a4/halftime/events/spectator
Publish topic: stadium/a4/halftime/metrics/kpi


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


In [ ]:
# Cell 4 purpose: Subscribe to spectator events and feed metrics aggregator.
received_events = []
published_kpi_payloads = []

def _on_message(client, userdata, msg):
    try:
        payload = json.loads(msg.payload.decode('utf-8'))
    except json.JSONDecodeError:
        return

    received_events.append(payload)
    record_spectator_event(state, payload)

mqtt_client.on_message = _on_message
mqtt_client.subscribe(spectator_topic, qos=1)
print('Subscription started. Waiting for incoming spectator events...')

Subscription started. Waiting for incoming spectator events...


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=

In [ ]:
# Cell 5 purpose: Run short loop, publish final KPI snapshot, and disconnect.
run_for_s = 30
start = time.time()
while time.time() - start < run_for_s:
    time.sleep(0.5)

run_id = state.active_run_id or (received_events[-1]['run_id'] if received_events else 'a4-no-events')
final_payload = finalize_kpi_payload(state=state, run_id=run_id, timestamp_s=900)

ok = False
publish_attempts = 0
while publish_attempts < 2 and not ok:
    publish_attempts += 1
    try:
        ok = mqtt.publish_json_checked(mqtt_client, kpi_topic, final_payload, qos=1)
    except RuntimeError:
        connector = getattr(mqtt_client, '_simcity_connector', None)
        if connector is not None:
            connector.disconnect()
        mqtt_client = mqtt.connect_mqtt(app_config.mqtt, client_id_suffix='metrics-agent')
        mqtt_client.on_message = _on_message
        mqtt_client.subscribe(spectator_topic, qos=1)

if ok:
    published_kpi_payloads.append(final_payload)

print(f'Received spectator events: {len(received_events)}')
print(f'Published KPI payloads: {len(published_kpi_payloads)}')
if published_kpi_payloads:
    print('Final KPI payload:')
    print(published_kpi_payloads[-1])
elif publish_attempts >= 2:
    print('Final KPI publish failed after one reconnect retry.')

connector = getattr(mqtt_client, '_simcity_connector', None)
if connector is not None:
    connector.disconnect()
    print('Disconnected from MQTT broker.')

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=

RuntimeError: Message publish failed: The client is not currently connected.

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=